# 🧠 CerebroSpheroid V5: Kaggle Cloud TPU High-Throughput Engine
### 105.9M V5 Small Architecture • 100% Loss-Bearing Contiguous Pretraining • Multi-TPU Local-SGD

Runs pure JAX high-throughput training across all 8 TPU cores (Kaggle TPU v5e-8 / TPU v3-8):
- **Pure JAX on Cloud TPU**: Native bfloat16 systolic MXU acceleration (~100,000–350,000 tok/s on TPU v5e / v3-8).
- **Hardware Auto-Tuning**: Automatically discovers and tunes peak micro-batches and remat policy on active hardware.
- **100% Loss-Bearing Contiguous Tokens**: Zero prompt-masking waste; unbroken 4,096-token sequence blocks.
- **SmolLM2 Pretraining Curriculum**: Four-stage FineWeb, Stack-Edu, FineMath/OWM, and Cosmopedia mixture.
- **Remote Cloud Sync**: Real-time checkpoint mirroring (`latest.pt`, `best.pt`) and loss tracking on `bioflo.in` (Option A ~229 MB weights-only fast upload).
- **Kaggle Preemption Guard**: 8.5-hour clean wall-clock guard to safely flush checkpoints before Kaggle 9h runtime expiration.

---
### ⚡ Kaggle Hardware Setting:
In the right sidebar under **Settings**:
- **Accelerator**: TPU VM v3-8 or TPU v5e-8
- **Internet**: Turn **ON** (required for dataset streaming and checkpoint sync)

## Step 1: Environment Setup & Hardware Verification
Enables transparent hugepages, verifies JAX TPU backend, and ensures required data streaming packages.

In [ ]:
# ==============================================================================
# ⚡ Step 1: Environment Setup & TPU Hardware Verification
# ==============================================================================
!sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled" 2>/dev/null || true
!pip install -q transformers "datasets>=3.0.0" sentencepiece optax torch pyarrow huggingface_hub

import os, sys
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TORCH_XLA_DISABLE_DISTRIBUTED"] = "1"

import jax
devices = jax.devices()
backend = jax.default_backend().upper()
print(f"JAX Version     : {jax.__version__}")
print(f"JAX Backend     : {backend}")
print(f"Active Devices  : {len(devices)} x {devices[0].device_kind if devices else 'none'}")
for idx, dev in enumerate(devices):
    print(f"  - Core [{idx}]: {dev}")

if "TPU" not in backend:
    raise RuntimeError("TPU accelerator not active! Enable TPU in Kaggle Settings -> Accelerator.")


## Step 2: Repository Synchronization & LLaMA Tokenizer
Pulls the latest codebase from GitHub into `/kaggle/working/cerebro_spheroid_v5`.

In [ ]:
# ==============================================================================
# 🔄 Step 2: Synchronize Repository to Latest origin/main
# ==============================================================================
import os, sys, subprocess, importlib

REPO_URL = "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git"
REPO_DIR = "/kaggle/working/cerebro_spheroid_v5"

if not os.path.exists(REPO_DIR):
    print(f"📦 Cloning repository {REPO_URL} -> {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Purge stale module caches
importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "kaggle")):
        del sys.modules[mod]

# Verify offline LLaMA tokenizer
tok_path = os.path.join(REPO_DIR, "data", "tokenizer")
assert os.path.isdir(tok_path), f"Tokenizer directory missing at {tok_path}"
print(f"🔤 Offline LLaMA-32k tokenizer verified at {tok_path}")


## Step 3: Authenticate Hugging Face & Bioflo Cloud Sync
Dynamically authenticates with Hugging Face Hub (eliminates 429 rate limits) and sets up bi-directional checkpoint streaming to `bioflo.in`.

In [ ]:
# ==============================================================================
# 🔑 Step 3: Dynamic Credential Authentication (Secrets & Website Vault)
# ==============================================================================
import os

# 1. Check Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_sec = secrets.get_secret("HF_TOKEN")
    if hf_sec:
        os.environ["HF_TOKEN"] = hf_sec
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_sec
    bioflo_sec = secrets.get_secret("BIOFLO_SYNC_TOKEN")
    if bioflo_sec:
        os.environ["BIOFLO_SYNC_TOKEN"] = bioflo_sec
        os.environ["CEREBRO_SYNC_TOKEN"] = bioflo_sec
except Exception:
    pass

# 2. Dynamic website vault fallback
from src.cloud_sync import get_sync_token
from src.hf_utils import get_hf_token

bioflo_tok = get_sync_token()
if bioflo_tok:
    os.environ["BIOFLO_SYNC_TOKEN"] = bioflo_tok
    os.environ["CEREBRO_SYNC_TOKEN"] = bioflo_tok
    print("🌐 Bioflo Cloud Sync Token successfully authenticated.")

hf_tok = get_hf_token()
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok
    try:
        from huggingface_hub import login
        login(token=hf_tok, add_to_git_credential=False)
        print(f"🔑 Hugging Face Hub successfully authenticated dynamically ({hf_tok[:4]}...{hf_tok[-4:]}).")
    except Exception as e_login:
        print(f"ℹ️ Hugging Face login notice: {e_login}")


## Step 4: Configure & Launch Pretraining
Launches contiguous pretraining for `v5_small` (105.9M params, 16 shells, 40 shortcut tracts).
- Automatically resumes from `bioflo.in` latest.pt checkpoint with smooth 100-step warmup.
- Automatically tunes batch size and rematerialization policy for TPU MXU.
- Safely checkpoints locally to `/kaggle/working/checkpoints` and streams ~229 MB weight snapshots to `bioflo.in`.

In [ ]:
# ==============================================================================
# 🚀 Step 4: Configure & Launch V5 Small High-Throughput Pretraining
# ==============================================================================
from training.train_v5_tpu_jax import run_training

CKPT_DIR = "/kaggle/working/checkpoints/v5_small_scratch"
os.makedirs(CKPT_DIR, exist_ok=True)

run_training(
    preset="v5_small",                     # 105.9M parameters, 16 concentric shells, 40 shortcut tracts
    from_scratch=False,                    # Resume from latest checkpoint on bioflo.in
    auto_tune=True,                        # Hardware-native micro-batch & remat auto-tuning
    force_tune=False,                      # Use cached winner or benchmark
    effective_batch_size=64,               # Global effective batch size across 8 TPU cores
    save_interval=250,                     # Checkpoint every 250 steps
    target_tokens=15_700_000_000,          # 15.7 Billion Token pretraining budget
    dataset_source="pretrain",             # SmolLM2 4-stage mixture (85% FineWeb 10BT + 15% The Stack v2 Code)
    elastic_window=False,                  # Pure contiguous 100% loss-bearing sequences
    thinking_cycles=4,                     # 4 subconscious deliberation cycles per token
    min_context_window=512,                # Stage 0: 512 tokens
    context_stage_steps=100_000,           # Curriculum stages: 512 -> 1024 -> 2048 -> 4096 (100k steps/stage)
    total_window=4096,                     # Max sequence budget
    sensory_position_steps=4096,           # Learned position horizon
    max_phase_steps=4096,                  # Learned phase clock horizon
    checkpoint_dir=CKPT_DIR,               # Kaggle scratch storage
    scratch_dir=CKPT_DIR,                  # Local staging directory
    cloud_sync=True,                       # Bioflo.in bi-directional sync
    cloud_weights_only=True,               # Option A: ~229 MB lightweight fast uploads (2.9s)
    save_optimizer_state=True,             # Preserves Spheroid moments locally on disk
    cloud_token=os.environ.get("BIOFLO_SYNC_TOKEN", ""),
    resume_target="latest",                # Resumes latest cloud slot
    resume_warmup_steps=100,               # Smooth 100-step warmup
    scmer_lambda=0.05,                     # SCMER anti-anisotropy regularizer strength
    scmer_margin=0.12,                     # Cosine margin threshold m
    scmer_subsample=512,                   # Subsampled vocabulary rows per event
    scmer_pair_samples=8192,               # Subsampled unique token pairs
    scmer_interval=8,                      # SCMER execution interval (every 8 steps)
    max_hours=8.5,                         # Clean exit before Kaggle 9-hour timeout
)
